# 03 — LLM Engine: Klasifikasi Kategori & Urgency
**SuaraLens** | Klasifikasi teks menggunakan LLM self-hosted via Ollama (Qwen3 8B).


## Setup Ollama

Sebelum menjalankan notebook ini:
1. Install Ollama: https://ollama.ai/download
2. Jalankan server: `ollama serve`
3. Pull model: `ollama pull qwen3:8b`

Di Colab, gunakan:
```bash
!curl -fsSL https://ollama.ai/install.sh | sh
!nohup ollama serve &
!sleep 5 && ollama pull qwen3:8b
```


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from modules.llm_engine import (
    classify_llm, classify_llm_batch,
    check_ollama_status, VALID_CATEGORIES, SYSTEM_PROMPT
)

sns.set_theme(style='whitegrid')
DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/classification_results.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')


## 1. Diagnostik Ollama Server

In [ ]:
status = check_ollama_status()
print('=== Status Ollama ===')
print(f'  Running        : {status["running"]}')
print(f'  Model tersedia : {status["available_models"]}')
print(f'  Model dipilih  : {status["recommended_model"]}')

if not status['running']:
    print()
    print('⚠ Ollama tidak berjalan!')
    print('  Jalankan di terminal: ollama serve')
    print('  Lalu pull model    : ollama pull qwen3:8b')


## 2. Review System Prompt

In [ ]:
print('=== System Prompt ===')
print(SYSTEM_PROMPT)
print()
print(f'=== Kategori Valid ({len(VALID_CATEGORIES)} kategori) ===')
for cat in VALID_CATEGORIES:
    print(f'  - {cat}')


## 3. Uji pada 20 Sampel

In [ ]:
# Ambil 20 sampel stratified (per kategori, minimal 1)
sample_df = (
    df.groupby('kategori_true', group_keys=False)
      .apply(lambda x: x.sample(max(1, min(2, len(x))), random_state=42))
      .sample(min(20, len(df)), random_state=42)
      .reset_index(drop=True)
)

print(f'Menjalankan klasifikasi pada {len(sample_df)} sampel...')
print('(Estimasi waktu: ~20-60 detik tergantung hardware)\n')

model_name = status.get('recommended_model')
llm_results = []
times = []

for i, row in sample_df.iterrows():
    start = time.time()
    result = classify_llm(row['teks_aduan'], model=model_name)
    elapsed = time.time() - start
    times.append(elapsed)
    llm_results.append({
        'id_aduan':        row['id_aduan'],
        'kategori_true':   row['kategori_true'],
        'kategori_pred':   result.get('kategori') if result else None,
        'urgency_true':    row['urgency_label_true'],
        'urgency_pred':    result.get('urgency_label') if result else None,
        'urgency_score':   result.get('urgency_score') if result else None,
        'confidence':      result.get('confidence') if result else None,
        'urgency_reason':  result.get('urgency_reason', '')[:80] if result else None,
        'inference_time':  round(elapsed, 2),
        'error':           result.get('error') if result else 'None returned',
    })
    print(f'  [{i+1}/20] {row["id_aduan"]} | pred={result.get("kategori") if result else "ERR"} | true={row["kategori_true"]}')

df_results = pd.DataFrame(llm_results)
print(f'\nSelesai. Avg inference time: {sum(times)/len(times):.2f}s / teks')


## 4. Hasil vs Ground Truth

In [ ]:
display(df_results[[
    'id_aduan', 'kategori_true', 'kategori_pred',
    'urgency_true', 'urgency_pred', 'confidence', 'inference_time'
]])

n_correct = (df_results['kategori_true'] == df_results['kategori_pred']).sum()
n_total   = len(df_results)
print(f'\nAkurasi pada 20 sampel: {n_correct}/{n_total} ({n_correct/n_total*100:.1f}%)')
print('(Ini hanya sanity check awal — evaluasi formal ada di NB 05)')


## 5. Benchmark Waktu Inference

In [ ]:
avg_time = sum(times) / len(times)
total_estimated = avg_time * len(df) / 60

print(f'=== Benchmark Inference ===')
print(f'  Sampel diukur   : {len(times)} teks')
print(f'  Rata-rata        : {avg_time:.2f} detik/teks')
print(f'  Estimasi total   : {total_estimated:.1f} menit untuk {len(df):,} baris')
print(f'  Model digunakan  : {model_name}')


## 6. Simpan Hasil ke JSON

In [ ]:
output = {
    'generated_at':     pd.Timestamp.now().isoformat(),
    'model_used':       model_name,
    'sample_size':      len(llm_results),
    'avg_inference_s':  round(sum(times) / len(times), 3),
    'results':          llm_results,
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)

print(f'Hasil disimpan ke: {OUTPUT_PATH}')
